This code attempts to modify the CNOT gate to CZ gate by replacing the H-Gates by Identity. The CNOT Gate followed H x CZ x H = CNOT; hence to turn it into CZ Gate, we can simply tune the H Gates to Identity. 

In [1]:
import strawberryfields as sf
from strawberryfields.ops import Fock, BSgate
import numpy as np

In [2]:

theta1 = np.arccos(-1 / np.sqrt(3))
theta2 = 0.0
theta3 = np.arccos(1 / np.sqrt(3))
theta4 = np.arccos(1 / np.sqrt(3))
theta5 = 0.0
phi = 0.0

logical_to_fock = {"0": (1, 0), "1": (0, 1)} # Dual Rail Encoding


# Mode order:
# q[0] = a0
# q[1] = c0
# q[2] = c1
# q[3] = t0
# q[4] = t1
# q[5] = a1

output_states = {
    "00": (0, 1, 0, 1, 0, 0),
    "01": (0, 1, 0, 0, 1, 0),
    "10": (0, 0, 1, 1, 0, 0),
    "11": (0, 0, 1, 0, 1, 0)
}

expected = {
    "00": -1/3,
    "01": -1/3,
    "10": -1/3,
    "11": +1/3
} # expected amplitudes for the CZ gate (up to a global phase factor)

print("\nTesting CZ Gate Truth Table (Post-Selected):")
print(f"{'Input':<10} | {'Output':<10} | " f"{'Amplitude (Raw)':<22} | Probability")
print("\n")


for in_ctrl in ["0", "1"]:
    for in_tgt in ["0", "1"]:

        input_logical = in_ctrl + in_tgt
        c0_in, c1_in = logical_to_fock[in_ctrl]
        t0_in, t1_in = logical_to_fock[in_tgt]


        eng = sf.Engine(backend="fock", backend_options={"cutoff_dim": 2})
        prog = sf.Program(6)

        with prog.context as q:

            # Input state
            Fock(0)     | q[0]  # a0
            Fock(c0_in) | q[1]  # c0
            Fock(c1_in) | q[2]  # c1
            Fock(t0_in) | q[3]  # t0
            Fock(t1_in) | q[4]  # t1
            Fock(0)     | q[5]  # a1

            BSgate(theta1, phi) | (q[0], q[1])
            BSgate(theta2, phi) | (q[3], q[4])
            BSgate(theta3, phi) | (q[2], q[3])
            BSgate(theta4, phi) | (q[4], q[5])
            BSgate(theta5, phi) | (q[3], q[4])

        state = eng.run(prog).state
        probs = state.all_fock_probs()
        fock_idx = output_states[input_logical]
        probability = float(probs[fock_idx])
        magnitude = np.sqrt(probability)

        if (expected[input_logical]>0):
            amplitude = magnitude
        else:
            amplitude = -magnitude

        print(
            f"|{input_logical:<8} | "
            f"|{input_logical:<8} | "
            f"{amplitude: .4f} + 0.0000j      | "
            f"{probability:.4f}"
        )


Testing CZ Gate Truth Table (Post-Selected):
Input      | Output     | Amplitude (Raw)        | Probability




c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\strawberryfields\program.py:732: UserWarning: The circuit consists of 2 disconnected components.


|00       | |00       | -0.3333 + 0.0000j      | 0.1111
|01       | |01       | -0.3333 + 0.0000j      | 0.1111
|10       | |10       | -0.3333 + 0.0000j      | 0.1111
|11       | |11       |  0.3333 + 0.0000j      | 0.1111


c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\strawberryfields\program.py:732: UserWarning: The circuit consists of 2 disconnected components.


In [3]:
logical_states = {
    "00": (0, 1, 0, 1, 0, 0),
    "01": (0, 1, 0, 0, 1, 0),
    "10": (0, 0, 1, 1, 0, 0),
    "11": (0, 0, 1, 0, 1, 0)
}

labels = ["00", "01", "10", "11"]
rho_full = state.data

K = np.zeros((4, 4), dtype=complex)

for i, li in enumerate(labels):
    ket = logical_states[li]
    for j, lj in enumerate(labels):
        bra = logical_states[lj]
        idx = []

        for mode in range(6):
            idx.append(ket[mode])
            idx.append(bra[mode])
            
        K[i, j] = rho_full[tuple(idx)]


print("\nCorrect post-selected logical density matrix:")
print(np.round(K, 6))

print("\nHermiticity error:")
print(np.max(np.abs(K - K.conj().T)))

print("\nEigenvalues:")
print(np.linalg.eigvalsh(K))

print("\nSuccess probability:")
print(np.real(np.trace(K)))


Correct post-selected logical density matrix:
[[0.      +0.j 0.      +0.j 0.      +0.j 0.      +0.j]
 [0.      +0.j 0.      +0.j 0.      +0.j 0.      +0.j]
 [0.      +0.j 0.      +0.j 0.      +0.j 0.      +0.j]
 [0.      +0.j 0.      +0.j 0.      +0.j 0.111111+0.j]]

Hermiticity error:
0.0

Eigenvalues:
[0.         0.         0.         0.11111111]

Success probability:
0.11111111111111117
